# Module X4: exercises

These use `monthly_demand.csv` (144 months of demand with a promotion flag). Each exercise is checked against values computed independently (library calls or hand arithmetic). Simulations are seeded; checks allow for Monte Carlo error.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Autocorrelation by hand.** Write `acf_by_hand(x, k)` returning the lag-`k` sample autocorrelation $r_k=\sum_{t=k+1}^{n}(x_t-\bar x)(x_{t-k}-\bar x)\,/\sum_{t}(x_t-\bar x)^2$.
Then store the lag-1 autocorrelation of the demand's **first difference** in `r1_diff` and the lag-12 autocorrelation of the demand's first difference in `r12_diff`.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf

dem = pd.read_csv(DATA / "monthly_demand.csv", parse_dates=["month"])
y = dem["demand"].to_numpy()

def acf_by_hand(x, k):
    pass   # YOUR CODE HERE

dy = np.diff(y)
r1_diff = r12_diff = None   # YOUR CODE HERE

In [ ]:
assert abs(acf_by_hand([1, 2, 3, 4, 5], 1) - 0.4) < 1e-12, "for 1,2,3,4,5 the lag-1 value is 4/10 = 0.4"
ref = acf(dy, nlags=12)
assert abs(r1_diff - ref[1]) < 1e-9 and abs(r12_diff - ref[12]) < 1e-9, "compare with statsmodels' acf"
print(f"✅ Correct! lag 1 = {r1_diff:.3f}, lag 12 = {r12_diff:.3f} (differencing removed the trend, but the yearly cycle remains)")

## Exercise 2

**Durbin–Watson by hand.** Write `durbin_watson_by_hand(e)` for a residual vector, $DW=\sum_{t\ge2}(e_t-e_{t-1})^2/\sum_t e_t^2$.
Fit OLS of `demand` on `years` (months/12) **only** (no promotion, no season) and store the Durbin–Watson statistic of its residuals in `dw_trend_only`. Is it above or below 2? Store `"below"` or `"above"` in `dw_side`.

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

def durbin_watson_by_hand(e):
    pass   # YOUR CODE HERE

years = np.arange(len(y)) / 12
dw_trend_only = None   # YOUR CODE HERE
dw_side = None         # "below" or "above"

In [ ]:
assert abs(durbin_watson_by_hand([1, -1, 1, -1]) - 3.0) < 1e-12, "alternating residuals: 3 squared steps of 2 over a sum of squares of 4 = 3"
fit_ref = sm.OLS(y, sm.add_constant(years)).fit()
assert abs(dw_trend_only - durbin_watson(fit_ref.resid)) < 1e-9
assert dw_side == "below", "the seasonal swings left in the residuals are positively autocorrelated"
print(f"✅ Correct! DW = {dw_trend_only:.2f}: strong positive autocorrelation (the seasonal pattern was left in the residuals)")

## Exercise 3

**Effective sample size.** Write `n_eff(n, phi)` $=n\,(1-\phi)/(1+\phi)$ and `se_inflation(phi)` $=\sqrt{(1+\phi)/(1-\phi)}$, the factor by which the true standard error of a mean exceeds the naive one.
Store `n_eff_100` for $n=100,\ \phi=0.8$ and `inflation_09` for $\phi=0.9$. Then set `answer` to the letter of the correct statement:

* **a)** a negative $\phi$ makes the effective sample size *smaller* than $n$
* **b)** a negative $\phi$ makes the effective sample size *larger* than $n$ (neighbours partly cancel out)
* **c)** $n_\text{eff}$ does not depend on $\phi$

In [ ]:
import math

def n_eff(n, phi):
    pass   # YOUR CODE HERE

def se_inflation(phi):
    pass   # YOUR CODE HERE

n_eff_100 = inflation_09 = None   # YOUR CODE HERE
answer = None   # "a", "b" or "c"

In [ ]:
assert abs(n_eff(100, 0) - 100) < 1e-12, "independent data: n_eff = n"
assert abs(n_eff_100 - 100 / 9) < 1e-9, "100 * 0.2 / 1.8 = 11.1"
assert abs(inflation_09 - math.sqrt(19)) < 1e-9, "sqrt(1.9 / 0.1) = 4.36"
assert answer == "b", "negative autocorrelation (zig-zag) makes the mean *more* precise than independent data would"
print(f"✅ Correct! n_eff = {n_eff_100:.1f} of 100; at phi = 0.9 standard errors are {inflation_09:.2f} times too small")

## Exercise 4

**Fit an AR(1) by least squares.** Write `fit_ar1(x)` that regresses $x_t$ on $(1, x_{t-1})$ with `np.linalg.lstsq` and returns `(c, phi)`. Apply it to the **first difference of the demand series** and store the results in `c_d, phi_d`.
Then simulate a series with `phi=0.7` using the supplied generator (`sim`) and store its estimate in `phi_sim`; it should be close to 0.7.

In [ ]:
import numpy as np
from statsmodels.tsa.ar_model import AutoReg

def fit_ar1(x):
    pass   # YOUR CODE HERE

dy = np.diff(y)
c_d = phi_d = None   # YOUR CODE HERE

rng = np.random.default_rng(11)
sim = np.zeros(2000)
for i in range(1, 2000):
    sim[i] = 0.7 * sim[i - 1] + rng.normal()
phi_sim = None   # YOUR CODE HERE

In [ ]:
ref = AutoReg(dy, lags=1, trend="c").fit().params
assert np.allclose([c_d, phi_d], ref, atol=1e-8), "compare with statsmodels' AutoReg"
assert abs(phi_sim - 0.7) < 0.05, phi_sim
print(f"✅ Correct! phi of the differenced demand = {phi_d:.3f} (weak: differencing removed most of the dependence); simulated phi = {phi_sim:.3f}")

## Exercise 5

**AR(1) forecasts.** Write `ar1_forecast(mu, phi, sigma, last, h)` returning `(mean, sd)` for the forecast `h` steps ahead:
$\mu+\phi^h(\text{last}-\mu)$ and $\sigma\sqrt{(1-\phi^{2h})/(1-\phi^2)}$.
For $\mu=50,\ \phi=0.5,\ \sigma=4,\ \text{last}=70$, store the 1-step and 3-step results in `f1` and `f3`, and the horizon at which the forecast mean first comes **within 1 unit of $\mu$** in `h_close` (search $h=1,2,\dots$).

In [ ]:
import math

def ar1_forecast(mu, phi, sigma, last, h):
    pass   # YOUR CODE HERE

f1 = f3 = None   # YOUR CODE HERE
h_close = None   # YOUR CODE HERE

In [ ]:
assert abs(f1[0] - 60) < 1e-9 and abs(f1[1] - 4) < 1e-9, "one step: mean 50 + 0.5*20 = 60, sd = sigma"
assert abs(f3[0] - 52.5) < 1e-9, "three steps: 50 + 0.125*20 = 52.5"
assert abs(f3[1] - 4 * math.sqrt((1 - 0.5 ** 6) / 0.75)) < 1e-9
assert h_close == 5, "20 * 0.5^h < 1 first happens at h = 5 (20/32 = 0.625)"
print(f"✅ Correct! 3-step forecast {f3[0]:.1f} with sd {f3[1]:.2f}; the forecast is within 1 of the mean from h = {h_close}")

## Exercise 6

**Baselines and MASE.** Train on the first 120 months and test on the last 24. Write `naive_forecast(train, h)` (repeat the last value) and `seasonal_naive_forecast(train, h, m=12)` (repeat the value from `m` months earlier, cycling). Write `mase(actual, forecast, train, m=12)` = MAE of the forecast divided by the MAE of the in-sample seasonal-naive forecast on the training data.
Store the MAE of each baseline in `mae_naive`, `mae_snaive` and their MASEs in `mase_naive`, `mase_snaive`.

In [ ]:
import numpy as np

train, test = y[:120], y[120:]

def naive_forecast(train, h):
    pass   # YOUR CODE HERE

def seasonal_naive_forecast(train, h, m=12):
    pass   # YOUR CODE HERE

def mase(actual, forecast, train, m=12):
    pass   # YOUR CODE HERE

mae_naive = mae_snaive = mase_naive = mase_snaive = None   # YOUR CODE HERE

In [ ]:
assert list(seasonal_naive_forecast(np.arange(24.0), 3)) == [12.0, 13.0, 14.0], "repeat the values from one year earlier"
assert list(naive_forecast(np.arange(5.0), 2)) == [4.0, 4.0]
assert abs(mae_naive - 9.89) < 0.01 and abs(mae_snaive - 10.75) < 0.01, (mae_naive, mae_snaive)
assert abs(mase_naive - 0.82) < 0.01 and abs(mase_snaive - 0.89) < 0.01, (mase_naive, mase_snaive)
print(f"✅ Correct! naive MAE {mae_naive:.2f} (MASE {mase_naive:.2f}); seasonal naive MAE {mae_snaive:.2f} (MASE {mase_snaive:.2f}); any serious model must beat these")

## Exercise 7

**Decomposition from scratch.** Write `centered_ma_12(x)` returning the centred $2\times12$ moving average (weights $(\tfrac12,1,\dots,1,\tfrac12)/12$, `NaN` for the first and last 6 values). Use it to compute the **12 seasonal indices** (monthly means of `y - trend`, shifted to sum to zero) as `season_idx` (January first). Then store the number of the month with the largest index in `peak_month`.

In [ ]:
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

def centered_ma_12(x):
    pass   # YOUR CODE HERE

trend = centered_ma_12(y)
season_idx = None    # array of 12 numbers
peak_month = None    # 1..12

In [ ]:
ref = seasonal_decompose(y, model="additive", period=12)
assert np.allclose(trend, ref.trend, equal_nan=True), "the trend should match statsmodels, including the NaN positions"
assert np.allclose(season_idx, ref.seasonal[:12]), "the seasonal indices should match statsmodels"
assert abs(season_idx.sum()) < 1e-9
assert peak_month == 6, "demand peaks in June"
print(f"✅ Correct! The peak is month {peak_month}; indices range from {season_idx.min():.1f} to {season_idx.max():.1f}")

## Exercise 8

**Naive vs robust standard errors.** Fit OLS of `demand` on a constant, `years`, `promo` and the 11 month dummies (drop January). Store the promotion coefficient's naive standard error as `se_naive`, its HAC standard error with `maxlags=6` as `se_hac`, and the ratio HAC/naive for the **`years`** coefficient as `ratio_years`.
Then set `answer` to the letter of the right statement:

* **a)** HAC standard errors are always larger than naive ones
* **b)** HAC standard errors matter most for slowly-moving regressors such as a trend, and can even be *smaller* for a randomly assigned regressor such as the promotion
* **c)** HAC corrects the coefficients as well as the standard errors

In [ ]:
import pandas as pd
import statsmodels.api as sm

X = pd.get_dummies(dem["month"].dt.month, prefix="m", drop_first=True).astype(float)
X.insert(0, "years", np.arange(len(y)) / 12)
X.insert(1, "promo", dem["promo"].astype(float))
X = sm.add_constant(X)

se_naive = se_hac = ratio_years = None   # YOUR CODE HERE
answer = None   # "a", "b" or "c"

In [ ]:
assert abs(se_naive - 1.444) < 0.005 and abs(se_hac - 1.223) < 0.005, (se_naive, se_hac)
assert abs(ratio_years - 1.456) < 0.01, ratio_years
assert answer == "b", "HAC changes only the standard errors, and not always upwards"
print(f"✅ Correct! promo SE {se_naive:.3f} -> {se_hac:.3f} (smaller); trend SE inflated by {ratio_years:.2f}x")

## Exercise 9

**Validation design.** You want an honest estimate of how well a demand-forecasting model will do next year. Which procedure is appropriate? Set `answer` to a letter.

* **a)** Shuffle the 144 months randomly, train on 80%, test on 20%
* **b)** Train on the earlier months and test on later ones (ideally repeating this with several cut-off dates, a "rolling origin")
* **c)** Fit on all 144 months and report the error on the same 144 months
* **d)** Train on the last 24 months and test on the first 120

In [ ]:
answer = None   # "a", "b", "c" or "d"

In [ ]:
assert answer == "b", "forecasting always means predicting the future from the past; shuffling leaks neighbouring months into training"
print("✅ Correct!")